## Tools with LangChain Agents

### Installing Utilities and Libraries

In [ ]:
%pip install langchain-anthropic==1.5.4

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
anthropic_model_name = os.getenv("ANTHROPIC_MODEL_NAME")

### Implement the Weather API Caller

In [ ]:
import requests
from typing_extensions import Annotated

def get_current_weather(
        location: Annotated[str, ..., "Location for which the weather condition needs to be fetched"]
    ):

    response = requests.get(
        f"https://wttr.in/{location}",
        params={
            "format": "j1"
        }
    )

    response.raise_for_status()

    return response.json()

### Instantiate the ChatAnthropic Class

In [ ]:
from langchain_anthropic import ChatAnthropic

model = ChatAnthropic(
    model_name = anthropic_model_name,
    api_key = anthropic_api_key,
)

### Bind the Model with Tools

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

model_with_tools = model.bind_tools([get_current_weather])

# User message
messages = [
    HumanMessage("What is the weather of London currently like?")
]

# First model call
response = model_with_tools.invoke(messages)

# Add assistant message containing tool call
messages.append(response)

# Execute every requested tool
for tool_call in response.tool_calls:

    if tool_call["name"] == "get_current_weather":

        result = get_current_weather(**tool_call["args"])

        messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"]
            )
        )

# Second model call (LLM now sees tool output)
final_response = model_with_tools.invoke(messages)

print(final_response.content)

### Create the MCP Server

In [ ]:
from anthropic.types.beta import BetaMCPToolsetParam

mcp_servers = [
    {
        "type": "url",
        "url": "https://learn.microsoft.com/api/mcp",
        "name": "Microsoft Learn",
    }
]

mcp_tool = BetaMCPToolsetParam(
    type="mcp_toolset",
    mcp_server_name="Microsoft Learn",
)

### Recreate the Model

In [ ]:
model = ChatAnthropic(
    model_name = anthropic_model_name,
    api_key = anthropic_api_key,
    mcp_servers = mcp_servers
)

### Invoke the Model 

In [ ]:
response = model.invoke(
    "Grab information about Microsoft Foundry from MS Learn MCP Server",
    tools = [mcp_tool]
)

print(response.text)